<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 03: Scale a study

**Goal:** Keep a two-stage MD study moving through one active `Batch`.

**Estimated time:** 15–20 minutes  
**Next step:** [Return to the course overview →](../../README.md)

<img src="assets/course-roadmap-module03.svg" alt="ALCHEMI Core roadmap with Fixed stages highlighted in Module 03" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:1rem 0 0;">

In [ ]:

import torch
from helpers import module03 as helpers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
if device.type == "cuda":
    print("gpu:   ", torch.cuda.get_device_name(device))

 Module 02 ran MD simulations on three selected complexes. Real screening studies can span hundreds, thousands, or even millions of systems. Here, weΓÇÖll learn the ropes with a fast study of a few hundred structures.

We will start from all 205 compatible NCIA250 `AB` complexes. First choose how many independent trajectories to prepare for each complex.

In [ ]:
batch_multiplier = 0.5  # positive scale factor for the 205-structure study

`batch_multiplier` scales the 205 structures. Set it to `0.5` for 103 trajectories, `1` for 205, or `2` for 410. Fractional results round to the nearest whole trajectory. The illustration and all three routes use the same resolved study size.

### Watch work move through the active `Batch`

The next cell uses the study size to display schematic NVT → NVE timing for the three routes. Watch when trajectories change stage and when waiting trajectories enter the active `Batch`.

In [ ]:
helpers.show_scheduling_widget(total_systems=205 * batch_multiplier)

## Route 1 · Sequential stages

Real simulation studies often move systems through several stages, such as temperature equilibration followed by production sampling. This lesson represents that pattern with a compact NVT → NVE workflow.

The simplest route processes one group at a time. The whole group completes NVT, then the whole group continues through NVE. Early finishers wait for the rest of their group, and the next group begins when both stages are complete.

Start by giving each trajectory its own 50 K velocity draw. The seed makes initialization reproducible.

In [ ]:
initial_temperature = 50.0  # K for each independent velocity draw
random_seed = 42  # reproducible velocity initialization

Now load the 205 structures, rebuild the Module 02 interaction model, and prepare the selected number of trajectories on `device`.

In [ ]:
trajectories, interaction_model = helpers.prepare_ncia250_trajectories(
    device,
    batch_multiplier=batch_multiplier,
    initial_temperature=initial_temperature,
    random_seed=random_seed,
)

`trajectories` holds the prepared trajectory records, `interaction_model` is shared by every route, and `device` selects where their tensors live.

### One study, two MD stages

Every trajectory starts from its own 50 K velocity draw. We will build NVT first, then continue the same positions and velocities with NVE. Each stage gets its timestep, duration cap, and stopping rule. 

`status = 0` runs NVT, `status = 1` runs NVE, and `status = 2` marks completion. The rolling rules let different trajectories change status at different times, giving the scheduling routes useful work to manage.

We will build three routes from the same prepared trajectories and the same two stage definitions:

1. fixed groups run NVT, then NVE;
2. `FusedStage` routes each trajectory by its own `status`; and
3. `FusedStage` with inflight batching fills released slots from waiting work.

Each route needs fresh stage objects because both stopping hooks carry per-trajectory GPU history. We will build the NVT and NVE stages separately, then combine them in `build_md_stages()`.

First choose the observation cadence shared by both stopping rules. With a 1 fs timestep, sampling every 10 steps makes each 10-sample window cover 100 fs. Two passing windows change a trajectory's status.

In [ ]:
sample_every = 10  # MD steps between screening samples
window_samples = 10  # 10 samples × 10 steps = 100 fs
required_windows = 2  # passing windows before a status change

Different trajectories equilibrate at different rates, so each needs its own stopping rule. Every 10 fs, the rule records the current temperature. A 100 fs window passes when its mean is within 60 K of 300 K and the means of its two halves differ by at most 45 K.

The next cell builds a fresh `temperature_window_hook()` with those choices. The hook keeps each trajectory's history on `device` and owns the status change from `0` to `1`.

In [ ]:
def build_temperature_hook():
    # Give each route independent 100 fs temperature histories.
    return helpers.temperature_window_hook(
        total_systems=len(trajectories),
        device=device,
        frequency=sample_every,
        window_samples=window_samples,
        required_streak=required_windows,
        target_temperature=300.0,  # K
        temperature_tolerance=60.0,  # K around the target
        trend_tolerance=45.0,  # K between half-window means
    )

Now construct NVT with a 1 fs timestep, a 300 K target, 0.01 fs⁻¹ Langevin friction, and a 600 fs cap. `build_temperature_hook()` gives every new stage independent rolling history, while `make_neighbor_hooks()` refreshes neighbors before force evaluation.

`FusedStage` normally adds a force-based transition between adjacent stages. This study uses the rolling temperature rule, so `keep_temperature_hook_in_control()` supplies the small adapter that leaves status `0 → 1` with that hook.

In [ ]:
from nvalchemi.dynamics import NVE, NVTLangevin


def build_nvt_stage():
    return NVTLangevin(
        model=interaction_model,
        dt=1.0,  # fs per Langevin step
        temperature=300.0,  # K
        friction=0.01,  # fs⁻¹ Langevin friction
        random_seed=random_seed,  # reproducible Langevin noise
        n_steps=600,  # 600 steps = 600 fs
        # Let the rolling temperature hook control status 0 → 1.
        convergence_hook=helpers.keep_temperature_hook_in_control(),
        hooks=[
            # Refresh neighbors before each standalone force evaluation.
            *interaction_model.make_neighbor_hooks(),
            # Store 100 fs temperature windows; change status 0 → 1.
            build_temperature_hook(),
        ],
    )

NVE continues from the same positions and velocities with 1 fs microcanonical steps and a 500 fs cap. Its rule samples total energy per atom every 10 fs and passes when the 100 fs window spans at most 0.00075 eV per atom.

The next cell creates a fresh energy history for each route and owns the change from status `1` to `2`.

In [ ]:
def build_energy_hook():
    # Give each route independent 100 fs energy histories.
    return helpers.energy_window_hook(
        total_systems=len(trajectories),
        device=device,
        frequency=sample_every,
        window_samples=window_samples,
        required_streak=required_windows,
        span_per_atom=0.00075,  # eV/atom across the window
    )

Now construct the NVE stage. `NVE` continues each status-`1` trajectory with the visible timestep and cap, refreshes neighbors before force evaluation, and gives the fresh energy hook ownership of status `1 → 2`.

In [ ]:
def build_nve_stage():
    return NVE(
        model=interaction_model,
        dt=1.0,  # fs per microcanonical step
        n_steps=500,  # 500 steps = 500 fs
        exit_status=2,  # status 2 marks a completed trajectory
        hooks=[
            # Refresh neighbors before each standalone force evaluation.
            *interaction_model.make_neighbor_hooks(),
            # Store 100 fs energy windows; change status 1 → 2.
            build_energy_hook(),
        ],
    )

Combine the two constructors in one small function. Every route calls it to receive independent stage objects and fresh rolling histories.

In [ ]:
def build_md_stages():
    # Return fresh NVT and NVE stages for one group or inflight route.
    return build_nvt_stage(), build_nve_stage()

### Choose the active `Batch` capacity

The NVT and NVE stages are ready. Route 1 now needs one active-`Batch` capacity. `max_batch_size` limits the number of systems, while `max_atoms` limits their combined atom rows. The same two limits will be reused by Routes 2 and 3.

In [ ]:
max_batch_size = 64  # systems in one active Batch
max_atoms = 1182  # atoms in one active Batch

The preparation step returned ordinary `trajectories` and `interaction_model` variables. Define one visible notebook function that clones those records with Toolkit's public `Batch` constructor. Every route will call the same function to start from independent state.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `Batch.from_data_list(data_list, device=device)`
>
> **Input:** cloned prepared trajectories and the lesson device  
> **Returns:** one new packed `Batch` with positions, velocities, energies, forces, and status on that device

In [ ]:
from nvalchemi.data import Batch


def fresh_batch():
    # Clone every prepared trajectory before packing the route's `Batch`.
    return Batch.from_data_list(
        [record.clone() for record in trajectories],
        device=device,
    )

Call the visible notebook function to create Route 1's independent state. Inspect `num_graphs` to confirm the full study and `device` to confirm where its tensors live.

In [ ]:
fixed_batch = fresh_batch()
print("fixed_batch.num_graphs:", fixed_batch.num_graphs)
print("fixed_batch.device:", fixed_batch.device)

`fixed_group_indices()` reads each graph's atom count and applies the active system and atom limits. It returns one index tensor per capacity-safe group. Inspect their lengths before packing any `Batch` objects.

In [ ]:
fixed_indices = helpers.fixed_group_indices(
    fixed_batch,
    max_batch_size=max_batch_size,
    max_atoms=max_atoms,
)
[len(indices) for indices in fixed_indices]

Pack each visible index tensor with the public `Batch.index_select()` method. Every call returns a new fixed group with all graph-, node-, and edge-level fields aligned. Inspect the group occupancies; they sum to the full study.

In [ ]:
fixed_groups = [
    fixed_batch.index_select(indices) for indices in fixed_indices
]
[group.num_graphs for group in fixed_groups]

`finish_fixed_stage()` makes every Toolkit `step()` visible. Each call updates the same `Batch` until all trajectories reach the target status or the stage cap. Before advancing cap-limited trajectories, it counts which trajectories passed the rolling rule naturally.

In [ ]:
def finish_fixed_stage(group, stage, target_status):
    # Each public `step()` updates this same GPU `Batch`.
    for _ in range(stage.n_steps):
        group, _ = stage.step(group)
        if bool((group.status >= target_status).all()):
            break

    passed_rule = int((group.status >= target_status).sum().item())
    advanced_at_cap = group.num_graphs - passed_rule

    # Advance cap-limited systems so the next stage can begin.
    group.status.clamp_min_(target_status)
    return group, passed_rule, advanced_at_cap

Now run Route 1. Each fixed group receives fresh NVT and NVE stages, completes both stages, and enters `fixed_results`. The four counters retain how many trajectories passed each rolling rule and how many reached each cap. `time_route()` synchronizes the device around the complete route.

In [ ]:
fixed_results = []
nvt_passed = nvt_capped = 0
nve_passed = nve_capped = 0

# Time the complete fixed route with CUDA synchronization.
with helpers.time_route(device) as fixed_run:
    for group in fixed_groups:
        # Give each group independent stages and rolling histories.
        nvt, nve = build_md_stages()
        group, passed, capped = finish_fixed_stage(group, nvt, 1)
        nvt_passed += passed
        nvt_capped += capped
        group, passed, capped = finish_fixed_stage(group, nve, 2)
        nve_passed += passed
        nve_capped += capped
        fixed_results.append(group)

Inspect how each stopping rule ended. A trajectory either passed the rolling rule during the stage or was advanced when the duration cap was reached.

In [ ]:
print("NVT passed rolling rule:", nvt_passed)
print("NVT advanced at 600 fs cap:", nvt_capped)
print("NVE passed rolling rule:", nve_passed)
print("NVE advanced at 500 fs cap:", nve_capped)

`summarize_fixed()` checks the completed groups against the expected trajectory identities and `device`, then combines them with the synchronized timer. Inspect the completion count, elapsed time, throughput, memory, and replacement count.

In [ ]:
fixed_result = helpers.summarize_fixed(
    fixed_results,
    fixed_run,
    expected_systems=len(trajectories),
    device=device,
)
fixed_result.summary

> **What to check:** `completed trajectories` equals `205 * batch_multiplier`. The rule and cap counts above show how many trajectories changed stage naturally. Fixed groups create two group-wide waiting points, so early finishers keep their slots until each group crosses the boundary.

## Route 2 · Fused stages

`FusedStage` keeps NVT and NVE in one active `Batch`. Each trajectory moves to NVE when its own NVT rule passes, while the others continue in NVT. The status map selects the MD update applied to each trajectory.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `nvt + nve`
>
> **Input:** two dynamics stages that share one model  
> **Returns:** a `FusedStage` with NVT at status `0` and NVE at status `1`

In [ ]:
from nvalchemi.dynamics import FusedStage

Toolkit uses `+` to fuse two dynamics stages that share one `Batch` and model evaluation. `register_hook()` attaches each model-owned neighbor hook to the new route, where it refreshes neighbor data before the shared model call. The function returns one independent route, and the next cell creates one for every fresh group.

In [ ]:
def build_fused_route():
    # Give each fixed group fresh stages and rolling histories.
    nvt, nve = build_md_stages()
    fused = nvt + nve  # status 0 = NVT; status 1 = NVE

    # Refresh neighbors before each shared fused model evaluation.
    for hook in interaction_model.make_neighbor_hooks():
        fused.register_hook(hook)

    return fused

Route 2 starts from another independent clone of the prepared trajectories. Call the visible `fresh_batch()` function and inspect the returned graph count and device before selecting any groups.

In [ ]:
fused_batch = fresh_batch()
print("fused_batch.num_graphs:", fused_batch.num_graphs)
print("fused_batch.device:", fused_batch.device)

Reuse Route 1's visible `fixed_indices` so both routes have identical membership. Each `Batch.index_select()` call returns a new group; inspect the occupancies after packing.

In [ ]:
fused_groups = [fused_batch.index_select(indices) for indices in fixed_indices]
[group.num_graphs for group in fused_groups]

Create one independent `FusedStage` for every prepared group. Each call to `build_fused_route()` creates fresh NVT and NVE stages with fresh rolling histories; the resulting list aligns one route with each group. Inspect the list length before pairing routes and groups.

In [ ]:
# Give each fixed group independent stages and rolling histories.
fused_routes = [build_fused_route() for _ in fused_groups]
len(fused_routes)

Pair each fresh group with its independent `FusedStage`. `fused.run(group)` updates and returns one completed `Batch`; the loop stores those batches inside the synchronized timer. The following summary checks their statuses and identities.

In [ ]:
fused_results = []

# Time the full fixed-membership fused route with CUDA synchronization.
with helpers.time_route(device) as fused_run:
    for group, fused in zip(fused_groups, fused_routes, strict=True):
        # `run()` updates and returns this completed GPU `Batch`.
        fused_results.append(fused.run(group))

`summarize_fused()` checks completed statuses and trajectory identities against the expected study size and `device`, then returns the same `RouteResult` fields as Route 1.

In [ ]:
# Check the completed Batches against the expected identities and timer.
fused_result = helpers.summarize_fused(
    fused_results,
    fused_run,
    expected_systems=len(trajectories),
    device=device,
)

# Display the checked route measurements.
fused_result.summary

The trace also starts from an independent clone outside the timing boundary. Create that `Batch` first and inspect its graph count and device.

In [ ]:
trace_batch = fresh_batch()
print("trace_batch.num_graphs:", trace_batch.num_graphs)
print("trace_batch.device:", trace_batch.device)

Reuse the first capacity-safe index set calculated in Route 1. `Batch.index_select()` returns one fixed-membership trace group; inspect its graph count before attaching the recorder.

In [ ]:
trace_group = trace_batch.index_select(fixed_indices[0])
trace_group.num_graphs

Attach a stage-membership recorder to the prepared trace group. `stage_trace_hook()` samples every 10 fused steps, `register_fused_hook()` attaches it to this route, and `run()` returns the completed `Batch` while the recorder keeps the intermediate status counts.

In [ ]:
# Build one fresh route and one GPU stage-membership recorder.
trace_route = build_fused_route()
stage_trace = helpers.stage_trace_hook(frequency=sample_every)

# Register the recorder, then run the unmodified group through NVT → NVE.
trace_route.register_fused_hook(stage_trace)
trace_group = trace_route.run(trace_group)

`summarize_stage_trace()` formats the first snapshots in which NVT and NVE are both active. Inspect a row where both trajectory counts are nonzero.

In [ ]:
helpers.summarize_stage_trace(stage_trace)

> **What to check:** at least one row contains both NVT and NVE trajectories. `FusedStage` removes the group-wide stage boundary. Completed trajectories still occupy slots until that fixed group drains.

## Route 3 · Fused stages with inflight batching

Route 3 keeps the per-trajectory NVT to NVE routing from Route 2 and adds waiting, active, and completed work. When a completed trajectory leaves the active `Batch`, the sampler supplies another trajectory within the same capacity limits. All trajectory state stays on CUDA.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `InMemoryDataset(fresh_batch())`
>
> **Input:** a new GPU-resident `Batch` from the visible notebook function  
> **Returns:** waiting systems retained in that in-memory `Batch` on the same device

Inspect the waiting count and resident device before building its sampler.

In [ ]:
from nvalchemi.data.datapipes import InMemoryDataset

# Keep the GPU-resident trajectories in place for indexed sampling.
waiting = InMemoryDataset(fresh_batch())
print("waiting trajectories:", len(waiting))
print("waiting device:", waiting.in_memory_batch.device)

`InMemoryDataset` keeps the complete waiting study as one resident `Batch` and serves individual systems by index. `SizeAwareSampler` uses those indices and system sizes to decide what enters the active `Batch`.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `SizeAwareSampler(waiting, max_batch_size=64, max_atoms=1182)`
>
> **Input:** the resident waiting dataset and active capacity limits  
> **Effect:** builds the initial active `Batch`, tracks which systems have entered the workflow, and supplies replacements that fit released capacity

`max_batch_size` limits the number of systems, while `max_atoms` limits their combined atom rows. For these differently sized complexes, whichever limit is reached first controls the active `Batch`. Inspect the stored limits before the sampler supplies any systems.

In [ ]:
from nvalchemi.dynamics import SizeAwareSampler

# Apply the same capacity used by the fixed groups.
sampler = SizeAwareSampler(
    waiting,
    max_batch_size=max_batch_size,
    max_atoms=max_atoms,
)
print("sampler system limit:", sampler.max_batch_size)
print("sampler atom limit:", sampler.max_atoms)

`FusedStage` writes graduates through the Toolkit `DataSink` interface. The focused `CudaCompletionSink` extension retains each selected `Batch` chunk on the same CUDA device, including the study's custom `simulation_id` and `status` fields. Construct it visibly and inspect its capacity and device before attaching it to the inflight route.

In [ ]:
completed = helpers.CudaCompletionSink(
    capacity=len(trajectories),
    device=device,
)

print("completed.capacity:", completed.capacity)
print("completed.device:", completed.device)

`nvt + nve` creates the two-stage route used above. Inflight batching also needs a sampler, completion sink, and refill cadence, so this route uses the full `FusedStage(...)` constructor to supply them. The model-owned neighbor hooks keep the changing active `Batch` current.

Checking every five fused updates balances frequent admission with the cost of changing the active `Batch`. Inspect `inflight_mode` to confirm that the sampler enabled refills.

In [ ]:
# Create fresh stages and histories for the inflight route.
nvt, nve = build_md_stages()

inflight = FusedStage(
    sub_stages=[(0, nvt), (1, nve)],  # status 0 = NVT, 1 = NVE
    sampler=sampler,  # supplies waiting trajectories to the active Batch
    sinks=[completed],  # receives completed GPU Batch chunks
    refill_frequency=5,  # fused updates between refill checks
    # Refresh neighbors before each shared fused model evaluation.
    hooks=interaction_model.make_neighbor_hooks(),
)
print("inflight mode:", inflight.inflight_mode)

`inflight.run()` asks the sampler to build the initial active `Batch`. At each refill check, completed trajectories move to the sink and the sampler selects unconsumed waiting systems that fit the released atom and edge capacity.

The timed cell stores the final return value in `remaining`. The next check expects `None`, meaning that the waiting work and active `Batch` have both drained.

In [ ]:
# Time study admission, fused updates, refills, and completion.
with helpers.time_route(device) as inflight_run:
    # `None` means both waiting and active GPU work drained.
    remaining = inflight.run()

In [ ]:
inflight_result = helpers.check_inflight_run(
    completed,
    remaining,
    inflight_run,
    expected_systems=len(trajectories),
    device=device,
    max_batch_size=max_batch_size,
)
inflight_result.summary

`check_inflight_run()` calls the standard sink `read()` method and receives one completed `Batch`. It verifies every trajectory identity exactly once on `device` and reports the replacement count. Pass the shared `max_batch_size` so the count uses the capacity chosen for all three routes. A study that fits inside the initial active `Batch` reports `0` replacements.

## Compare the three routes

`compare_routes()` combines the three checked `RouteResult` objects into one table. Each row covers the same trajectories, model, initial state, MD settings, stopping rules, and active capacity.

The timer includes each route exactly as called here. `FusedStage.run()` refreshes initial forces inside its timed route; the fixed route starts from the forces prepared with the trajectories. Inspect study scope first, then time, throughput, memory, and replacements.

In [ ]:
helpers.compare_routes(
    fixed_result, fused_result, inflight_result
)

> **Bottom line**
>
> `FusedStage` chooses the algorithm for each trajectory. Inflight batching chooses which trajectories occupy the active `Batch`. Together they turn natural stopping-time variation into a continuously supplied GPU workflow.

Docs: [Multi-stage pipeline example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/01_multistage_pipeline.html) · [Inflight batching example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/04_inflight_batching.html)

Read `completed trajectories` first to confirm equal study scope. Then compare synchronized time and throughput. `replacement trajectories` reports how many systems entered after the initial active `Batch`. A value of `0` means the whole study fit at once; use a multiplier that creates more than 64 trajectories to watch refills.

These single live runs describe this study on the selected GPU. Repeated measurements are required for a speedup claim.



## Continue to multiple GPUs

This module used one GPU for many independent trajectories. The same Toolkit workflow could extend across GPUs through three execution layouts. And if your simulate large molecules - tens of thousands of atoms beyongd limits of a single GPU - check out **Domain decomposition** feature. 

<img src="assets/03-multi-gpu-continuations-v1.svg" alt="The Module 03 one-GPU campaign followed by three Toolkit multi-GPU layouts: DistributedPipeline streams systems through stage-specific GPU ranks, DomainParallel divides one large system among spatial domains, and pipeline × domain assigns a group of domain ranks to each stage." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.8rem 0 0.55rem;">

Examples: [Distributed FIRE → Langevin pipeline](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/01_distributed_pipeline.html) · [Domain-decomposed MACE NVT](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/03_mace_nvt_distributed.html) · [2-D pipeline × domain decomposition](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/07_fire_nvt_dd.html)



You have completed ALCHEMI Core Tutorial! 🎉

**Next step:** ~~[Return to the course overview →](../../README.md)~~   **CREATE**, **INNOVATE**, **DISCOVER** with Toolkit